<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_107_transformer_architecture_ablations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🏗️ **Module 107 — Modern Transformer Architecture Choices · Ablation Workshop** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 16 · Architecture Ablations (Imad Saddik Train-Your-LLM deep-dives)


# Module 107 — Modern Transformer Architecture Choices · Ablation Workshop

> M19/M20 built attention from scratch. M61 walked GPT-2 → Llama 3. **This module sits all the architecture choices side-by-side and ablates them under one budget.** Built around Imad Saddik's `Train_Your_Language_Model_Course` Chapter 9 (22 notebooks) and modernized to 2026: **positional encoding** (Absolute · Sinusoidal · Relative · RoPE · ALiBi · NoPE), **attention** (MHA · MQA · GQA · MLA · Local · BigBird · Linear · FlashAttention-3), **activation** (GELU · SwiGLU · GeGLU · ReGLU), **normalization** (LayerNorm · RMSNorm · DyT · Pre-/Post-/Sandwich-LN), and the **iterative best-model phases** that show how Llama, Mistral, DeepSeek, and Qwen ended up where they are.

### What you'll cover
1. The ablation framework — fix dataset, schedule, budget
2. **Positional encoding** — Abs · Sinusoidal · Relative · RoPE · ALiBi · NoPE
3. **Attention** — MHA · MQA · GQA · MLA · Local · BigBird · Linear · FlashAttention-3
4. **KV-cache implications** for each attention variant
5. **Activations** — GELU · SwiGLU · GeGLU · ReGLU · Mish
6. **Normalization** — LayerNorm · RMSNorm · DyT · Pre-/Post-/Sandwich-LN
7. **Iterative best-model phases 1-4** — stacking the winners (Imad 9.5)
8. **Loss curves + tok/s benchmarks** (Imad 9.6 + 9.7)
9. The **2026 frontier picker** — which choices Llama-3 / Mistral / DeepSeek / Qwen converged on
10. When to run an architecture ablation in your own project


## 1 · The ablation framework — fixing what doesn't change

The whole point of Imad's Chapter 9 is **scientific control**. To compare two architectural choices fairly:

```
FIXED:
  Dataset (same tokens, same order)
  Tokenizer + vocab size
  Total compute budget (tokens × parameters)
  Optimizer + LR schedule + warmup
  Random seed
VARIABLE (one at a time):
  Position encoding  -or-  attention variant  -or-  activation  -or-  norm
```

Without this discipline you'll attribute gains to architecture that actually came from a better LR or a longer warmup. **Most public LLM-architecture claims are confounded by exactly this.**

A good ablation reports:
- **Final val PPL** (or held-out NLL)
- **Loss curve** through training (early dynamics matter)
- **Tokens/sec** on the same hardware (M99 callback)
- **Memory** at training and inference
- **Quality on a downstream eval** (MMLU-Lite, HellaSwag — small set, fast)

> 🧪 **Imad's recipe** is a ~30M-parameter model trained for ~10M tokens on a single GPU — small enough to ablate dozens of choices in a few hours, big enough to see real differences. This is the right *teaching* scale; production ablations happen at 1-10B.


## 2 · Positional encoding — 6 variants

Transformers are permutation-invariant by default. Position encoding tells the model "this token came at position `t`." Six families:

| Variant | Formula | Length generalization | Used in |
|---|---|---|---|
| **Absolute (learned)** | `pos_emb[t]` added to token embed | Bad past training length | original GPT-2, BERT |
| **Sinusoidal** | `sin/cos(t / 10000^{2i/D})` | OK; some extrapolation | original Transformer (Vaswani 2017) |
| **Relative (Shaw 2018)** | Learned offset bias `R[i-j]` in attention | Better than absolute | T5, Transformer-XL |
| **RoPE** (Su 2021) | Rotate Q/K by angle `t·θ_i` | Strong w/ NTK / YaRN scaling | **Llama, Mistral, Qwen, DeepSeek** |
| **ALiBi** (Press 2022) | Linear bias `−m·|i−j|` on scores | Excellent extrapolation | MPT, BLOOM, Falcon |
| **NoPE** (Kazemnejad 2024) | None — let causal masking handle order | Surprisingly fine for some setups | Research |

**RoPE intuition.** Take Q and K vectors, treat consecutive pairs as 2D points, rotate each pair by an angle proportional to `position × frequency`. Same rotation applied to both Q and K means `q · k` depends only on their *relative* offset — automatic translation equivariance.

```python
def rope(x, theta=10000):
    # x: [B, H, T, D]
    D = x.shape[-1]
    freqs = theta ** (-torch.arange(0, D, 2) / D)         # [D/2]
    t = torch.arange(x.shape[-2])                         # [T]
    angles = t[:, None] * freqs[None, :]                  # [T, D/2]
    cos, sin = angles.cos(), angles.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin,
                        x1 * sin + x2 * cos], dim=-1).flatten(-2)
```

**2026 frontier choice:** RoPE for nearly everything (Llama-3, Mistral, Qwen-2.5, DeepSeek-V3). NTK-aware scaling + YaRN extends it to 128k-1M context.


In [ ]:
# Three positional encodings as drop-in functions over Q/K
import torch, math

def sinusoidal_pe(T, D, device='cpu'):
    pos = torch.arange(T, device=device)[:, None]                # [T, 1]
    i   = torch.arange(0, D, 2, device=device)[None, :]          # [1, D/2]
    div = torch.exp(-math.log(10000.0) * i / D)
    pe  = torch.zeros(T, D, device=device)
    pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe                                                    # add to token embed

def rope(x, base=10000.0):                                       # x: [B, H, T, D_head]
    D = x.shape[-1]
    freqs  = base ** (-torch.arange(0, D, 2, device=x.device) / D)
    angles = torch.arange(x.shape[-2], device=x.device)[:, None] * freqs[None, :]
    cos, sin = angles.cos(), angles.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)                                   # apply BEFORE dot product

def alibi_bias(H, T, device='cpu'):                              # added to attn scores
    slopes = 2 ** -torch.arange(1, H + 1, device=device) * 8 / H
    dist   = -(torch.arange(T)[:, None] - torch.arange(T)[None, :]).abs()
    return (slopes[:, None, None] * dist.to(device)).unsqueeze(0)  # [1, H, T, T]


## 3 · Attention variants — 8 architectures

This is the deepest ablation chapter. Imad covers 6 attention types; we add FlashAttention-3 (the 2024 kernel that makes any of them fast) and recent MLA from DeepSeek-V3.

| Variant | KV size per token | Speed | Used in |
|---|---|---|---|
| **MHA** (Vaswani 2017) | `H · d_k` (full) | Baseline | original Transformer |
| **MQA** (Shazeer 2019) | `d_k` (one K/V shared by all Q heads) | Big inference speedup, some quality drop | PaLM, Falcon |
| **GQA** (Ainslie 2023) | `G · d_k` (G < H groups share K/V) | Sweet spot — quality of MHA, speed of MQA | **Llama-3, Mistral, Qwen-2.5, Gemma-3** |
| **MLA** (DeepSeek-V3 2024) | Low-rank latent `c_KV ∈ R^{D_C}` (D_C ≪ D) | Tiny KV cache + restored quality via decoupled RoPE | DeepSeek-V2/V3, **2026 standout** |
| **Local / Sliding Window** (Beltagy 2020, Mistral 2023) | Last `W` tokens only | Linear context, predictable cache | Mistral-7B (W=4096), Longformer |
| **BigBird** (Zaheer 2020) | Local + random + global tokens | O(T) sparse | BigBird research; superseded |
| **Linear (Performer, Linformer)** (Choromanski 2020, Wang 2020) | None (kernelized) | True O(T·D²) | Speech / very long context, research |
| **FlashAttention-3** (Shah 2024) | Same as MHA but **block-tiled IO-aware** | 2-4× faster, fp8 supported | All modern LLM training |

**The KV-cache story** ties them all together. At inference, each new token reads ALL prior K/V. Memory = `2 · T · H · d_k · L · bf16` per request. For Llama-3-70B at 32k context, that's **~40 GB/request just for KV cache**. MQA / GQA / MLA exist to shrink this number.

**MLA's trick (DeepSeek-V3).** Cache only a low-rank latent `c_t = W_DKV · h_t ∈ R^{D_C}` (`D_C ≈ 512`), and **decouple** the RoPE-rotated portion so position info isn't compressed. At decode time `K_t = W_UK · c_t`, `V_t = W_UV · c_t` reconstructed cheaply. Result: **KV cache 1/16 of MHA's, quality matches MHA, decode 3-5× faster** — the reason DeepSeek-V3 can run 671B-MoE inference at the cost of a 30B dense model.

> 🚀 **2026 frontier choice:** **GQA with 8 KV groups** is the open default (Llama-3, Mistral, Qwen, Gemma). **MLA is the breakout** of 2024 — every long-context inference team is studying it.


In [ ]:
# MHA → GQA → MQA in one parameterized module (Llama-3-style)
import torch, torch.nn as nn, torch.nn.functional as F

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_kv_heads=2):    # n_kv=8 → MHA, n_kv=1 → MQA
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.h, self.h_kv = n_heads, n_kv_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, n_heads    * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x).view(B, T, self.h,    self.d_head).transpose(1, 2)   # [B, H, T, d]
        k = self.W_k(x).view(B, T, self.h_kv, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.h_kv, self.d_head).transpose(1, 2)
        k = k.repeat_interleave(self.h // self.h_kv, dim=1)                  # group share
        v = v.repeat_interleave(self.h // self.h_kv, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)        # FA-3 kernel
        return self.W_o(out.transpose(1, 2).reshape(B, T, -1))


In [ ]:
# MLA — DeepSeek-V3 multi-head latent attention (KV cache 1/16 of MHA)
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_latent=128, d_rope=32):
        super().__init__()
        self.h, self.d_head = n_heads, d_model // n_heads
        self.d_latent, self.d_rope = d_latent, d_rope
        # Down-projection for KV compression
        self.W_DKV  = nn.Linear(d_model, d_latent, bias=False)
        # Up-projections to recover full K/V at decode (no cache needed)
        self.W_UK   = nn.Linear(d_latent, n_heads * (self.d_head - d_rope), bias=False)
        self.W_UV   = nn.Linear(d_latent, n_heads *  self.d_head,           bias=False)
        # Decoupled RoPE-rotated K (cached separately; small)
        self.W_KR   = nn.Linear(d_model, d_rope, bias=False)
        self.W_Q    = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.W_o    = nn.Linear(d_model, d_model, bias=False)
    def kv_cache(self, x):                                         # store ONLY these per token
        return self.W_DKV(x), self.W_KR(x)                         # ~D_latent + D_rope ≪ H·d


## 4 · KV-cache implications

Per-variant KV size for a 70B-class model at 32k context, fp16:

| Variant | KV per token | Total at 32k |
|---|---|---|
| MHA (64 heads, d=128) | `64 · 256 = 16 KB` | **~16 GB** |
| GQA (8 KV groups, d=128) | `8 · 256 = 2 KB` | **~2 GB** |
| MQA | `1 · 256 = 256 B` | ~256 MB |
| MLA (`D_C=512`, `D_R=64`) | `~1.1 KB` | **~1.1 GB** (quality of MHA) |

**Why this dominates production inference.** A 70B dense model fits in 140 GB. With 16 GB/request KV-cache (MHA), you can fit 8 concurrent users on an H100 cluster. With MLA (1.1 GB/request), **you can fit ~100 concurrent users on the same cluster.** This is why DeepSeek-V3 API pricing is 10× cheaper than equivalent-quality OpenAI models — the math is real.

Production batching systems (vLLM, SGLang, TensorRT-LLM) explicitly track KV-cache memory; PagedAttention (Kwon 2023) and **FlashInfer** further reduce fragmentation.

> 🔁 **The 2024-2026 trend.** Frontier labs are uniformly moving toward attention variants that compress KV. After GQA (2023) the next step is either MLA (DeepSeek path) or **state-space models** (M84 — Mamba, RWKV) that have *constant* memory regardless of context length.


## 5 · Activations — what fires after `W_up`

The MLP's activation (M106 callback) gets less attention than it deserves. Six common choices:

| Activation | Formula | Smooth? | Used in |
|---|---|---|---|
| **ReLU** | `max(0, x)` | No | early Transformers, sparse-MLP routing |
| **GELU** (Hendrycks 2016) | `x · Φ(x)` ≈ `0.5x(1+tanh(...))` | Yes | GPT-2, BERT, GPT-3, T5 |
| **SwiGLU** (Shazeer 2020) | `(W_a x) ⊙ silu(W_b x)` (gated; doubles `W_up` params) | Yes | **Llama, Mistral, Qwen, Gemma, DeepSeek** |
| **GeGLU** | `(W_a x) ⊙ gelu(W_b x)` | Yes | PaLM |
| **ReGLU** | `(W_a x) ⊙ relu(W_b x)` | No | research |
| **Mish** | `x · tanh(softplus(x))` | Yes | niche |

**SwiGLU's secret** is the *gating* — the GLU (gated linear unit) structure. The model can choose, per feature, whether to pass through (gate ≈ 1) or suppress (gate ≈ 0). The doubled `W_up` parameter count is offset by reducing `D_FF` proportionally; the empirical result is **+1-2 PPL points** over plain GELU at the same parameter count.

**2026 frontier choice:** SwiGLU for any model > 1B parameters. GELU for very small / edge models where the gate's extra parameter cost matters (Phi-3.5-mini still uses GELU at 3.8B).


In [ ]:
# GPT-2 FFN (GELU) vs Llama FFN (SwiGLU) — same param budget
import torch.nn as nn, torch.nn.functional as F

class GeluFFN(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):                  # 4x expansion
        super().__init__()
        self.w_up   = nn.Linear(d_model, d_ff,   bias=True)
        self.w_down = nn.Linear(d_ff,    d_model, bias=True)
    def forward(self, x):
        return self.w_down(F.gelu(self.w_up(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, d_model=512, d_ff=1366):                  # 2/3 × 4 × D to match params
        super().__init__()                                       # SwiGLU has 3 matrices not 2
        self.w_gate = nn.Linear(d_model, d_ff,   bias=False)     # the GATE branch
        self.w_up   = nn.Linear(d_model, d_ff,   bias=False)
        self.w_down = nn.Linear(d_ff,    d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))  # gated linear unit


## 6 · Normalization — what tames the residual stream

Five normalization choices, two architectural questions:

**Where to normalize:**
| Layout | Order | Used in |
|---|---|---|
| **Post-LN** (Vaswani 2017) | `out = LN(x + sublayer(x))` | original Transformer, BERT |
| **Pre-LN** (Xiong 2020) | `out = x + sublayer(LN(x))` | GPT-2 onward (the modern default) |
| **Sandwich-LN** | LN before AND after sublayer | NormFormer |
| **DeepNorm** (Wang 2022) | Pre-LN with scaled residual | Microsoft Turing |

**What normalization:**
| Norm | Formula | Used in |
|---|---|---|
| **LayerNorm** (Ba 2016) | `(x − μ) / σ · γ + β` per token | GPT-2, BERT, T5 |
| **RMSNorm** (Zhang 2019) | `x / RMS(x) · γ` (no mean subtraction, no bias) | **Llama, Mistral, Qwen, Gemma, DeepSeek** |
| **DyT** (Zhu 2025 — Dynamic Tanh) | `tanh(α · x) · γ + β`, no statistics | research-grade, may replace LN at scale |

**Pre-LN won** because Post-LN is unstable for deep nets without careful warmup. Sandwich-LN gives marginal stability gains at 2× normalization cost.

**RMSNorm beat LayerNorm** for one reason: **fewer ops** (no mean computation, no bias term) at no quality cost. Saves ~1-2% of total inference compute — huge at scale.

**DyT (Zhu 2025, ICLR)** removed normalization entirely with `y = γ · tanh(α · x) + β` and matched LN quality. Not yet production but the most interesting normalization paper since RMSNorm.

> 🧯 **2026 frontier choice:** **Pre-LN + RMSNorm**. Universal across Llama-3, Mistral, Qwen-2.5, Gemma-3, DeepSeek-V3.


In [ ]:
# LayerNorm vs RMSNorm — and a transformer block in both Pre-LN and Post-LN layouts
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, D, eps=1e-5):
        super().__init__()
        self.g, self.b = nn.Parameter(torch.ones(D)), nn.Parameter(torch.zeros(D))
        self.eps = eps
    def forward(self, x):
        mu  = x.mean(-1, keepdim=True)
        var = x.var (-1, keepdim=True, unbiased=False)
        return (x - mu) / (var + self.eps).sqrt() * self.g + self.b

class RMSNorm(nn.Module):                                        # no mean, no bias = 2 ops saved
    def __init__(self, D, eps=1e-6):
        super().__init__()
        self.g = nn.Parameter(torch.ones(D))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.g

class PreLNBlock(nn.Module):                                     # modern (Llama-3)
    def __init__(self, attn, ffn, D):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(D), RMSNorm(D)
        self.attn, self.ffn    = attn, ffn
    def forward(self, x):
        x = x + self.attn(self.norm1(x))                         # NORM INSIDE the residual
        x = x + self.ffn (self.norm2(x))
        return x


## 7 · Best-model iterative phases — stacking winners

Imad's section 9.5 is the **constructive** part: take baseline GPT-2-style architecture, swap one choice at a time, keep the winner, advance. After 4 phases you've reconstructed the modern Llama-class architecture.

```
Phase 0 (baseline):  Abs-PE + MHA + GELU + Post-LN + LayerNorm + bias terms
   ↓ swap PE
Phase 1:             RoPE + MHA + GELU + Pre-LN + LayerNorm
   ↓ swap attention
Phase 2:             RoPE + GQA(8) + GELU + Pre-LN + LayerNorm
   ↓ swap activation
Phase 3:             RoPE + GQA + SwiGLU + Pre-LN + LayerNorm
   ↓ swap norm + remove biases
Phase 4:             RoPE + GQA + SwiGLU + Pre-LN + RMSNorm + no biases   ←  Llama-3 architecture
```

Imad's empirical PPL curve (small-scale, illustrative):
```
Phase 0:  PPL = 22.4
Phase 1:  PPL = 20.8   (RoPE)
Phase 2:  PPL = 20.5   (GQA — quality nearly held)
Phase 3:  PPL = 19.1   (SwiGLU — biggest single win)
Phase 4:  PPL = 18.7   (RMSNorm — speed boost, slight quality)
```

The lesson: **the modern architecture is the cumulative result of small, additive wins**, each ~5-10% PPL, none individually revolutionary. RoPE is the largest single jump, SwiGLU second. GQA is **break-even on quality** but huge on inference speed (Section 4).

This is also how every frontier-lab architecture paper now reports its work — "we changed X, kept everything else, see Y % improvement." Reading these papers with the ablation table in mind is the right way.


In [ ]:
# Phase config — flip ONE flag at a time, keep everything else fixed
PHASES = {
    "phase_0": dict(pe="absolute",   attn="MHA", act="GELU",   norm="LayerNorm",
                    layout="post_ln", bias=True),
    "phase_1": dict(pe="rope",       attn="MHA", act="GELU",   norm="LayerNorm",
                    layout="pre_ln",  bias=True),
    "phase_2": dict(pe="rope",       attn="GQA", act="GELU",   norm="LayerNorm",
                    layout="pre_ln",  bias=True),
    "phase_3": dict(pe="rope",       attn="GQA", act="SwiGLU", norm="LayerNorm",
                    layout="pre_ln",  bias=True),
    "phase_4": dict(pe="rope",       attn="GQA", act="SwiGLU", norm="RMSNorm",
                    layout="pre_ln",  bias=False),                # ← Llama-3 architecture
}

def build_model(cfg, d_model=256, n_layers=6, vocab=8192):
    attn_cls = {"MHA": (8, 8), "GQA": (8, 2), "MQA": (8, 1)}[cfg["attn"]]
    norm_cls = {"LayerNorm": LayerNorm, "RMSNorm": RMSNorm}[cfg["norm"]]
    ffn_cls  = {"GELU": GeluFFN, "SwiGLU": SwiGLUFFN}[cfg["act"]]
    # one-arg variants of attn/ffn omitted for brevity — same idea as Section 3 & 5
    return f"Built {cfg}"                                         # placeholder

for name, cfg in PHASES.items():
    print(name, "→", build_model(cfg))


## 8 · Loss curves + tok/s benchmarks

Imad's 9.6 + 9.7 are the **reporting** part. Two plots that everyone should make for every architecture ablation:

**Loss-curve plot** — overlay phases 0-4 on the same axes (steps vs val PPL).

What you'll see:
- Early training (steps 0-1k) is dominated by **LayerNorm choice** and **warmup**
- Mid training (steps 1k-10k) is where **architecture differences** appear — RoPE pulls ahead, GQA matches MHA
- Late training is where **SwiGLU's gating** consolidates

**Tokens/sec benchmark** — same hardware, same batch, same context length. Reports:
- Train tok/s (forward + backward)
- Inference tok/s (decode at batch 1, the user-facing latency)
- Memory at training + at inference (KV cache)

Typical small-model numbers (single A100, 80M params):
```
Phase 0 (Abs + MHA + GELU + Post-LN):  Train 45k tok/s,  Decode 220 tok/s
Phase 4 (RoPE + GQA + SwiGLU + RMS):    Train 52k tok/s,  Decode 380 tok/s
```

Note: **inference speedup is bigger than training**, because GQA / RMSNorm savings compound at decode (autoregressive, one token at a time).

> 📊 **A practical note from production.** When you read a frontier-lab paper claiming "+10% over the baseline," check whether they're reporting **same-FLOPs**, **same-parameters**, or **same-wall-clock**. They're all valid but not comparable. Imad's ablation framework forces you to pick one and report all three.


## 9 · The 2026 frontier picker

What the major open-source labs converged on:

| Model | PE | Attention | Activation | Norm | Bias |
|---|---|---|---|---|---|
| **Llama-3 / 3.1 / 3.2 / 3.3** | RoPE (+YaRN) | GQA-8 | SwiGLU | Pre-RMSNorm | No |
| **Mistral / Mixtral** | RoPE | GQA-8 + Local SW (4096) | SwiGLU | Pre-RMSNorm | No |
| **Qwen-2.5 / 3** | RoPE (NTK) | GQA-8 | SwiGLU | Pre-RMSNorm | No |
| **Gemma-3** | RoPE | GQA + Local-Global alt | GeGLU | Pre-RMSNorm | No |
| **DeepSeek-V3** | RoPE (decoupled) | **MLA** | SwiGLU + **MoE** (256 experts, top-8) | Pre-RMSNorm | No |
| **Phi-3.5 (edge)** | RoPE | MHA | **GELU** (still — small model) | Pre-LayerNorm | No |
| **DBRX** | RoPE | GQA | SwiGLU + MoE | Pre-RMSNorm | No |
| **Falcon-2** | RoPE | MQA | SwiGLU | Pre-RMSNorm | No |

**Convergence is striking.** Almost every model from 2024-2026 uses **RoPE + GQA + SwiGLU + Pre-RMSNorm + no biases**. The variations are at the *attention* tier:
- **MLA** (DeepSeek) for cheap long-context inference
- **Sliding-window** (Mistral) for predictable context cost
- **Local-Global alternation** (Gemma) for efficiency
- Vanilla **GQA** (Llama / Qwen) as the safe baseline

**The 2026 default** for any new pretraining is **the Llama-3 stack**: RoPE + GQA-8 + SwiGLU + Pre-RMSNorm + no biases + FlashAttention-3. Deviate only if you have a specific reason (e.g., MLA for ultra-cheap inference).

> 🌟 **Open question for 2026.** Will **MLA** become universal? It quietly delivers 10×-class inference cost reduction with no quality regression. The reason it hasn't taken over is partly inertia, partly that the *decoupled* RoPE implementation is fiddly. By late 2026 we'll see whether DeepSeek's choice generalizes.


## 10 · When to run an architecture ablation in your own project

Most ML practitioners shouldn't ablate architecture — **use the Llama-3 stack and spend your compute on data + scale**. Architecture ablation is for:

1. **You're publishing research.** Run ablations on every claim — without them, reviewers will reject.
2. **You're targeting unusual constraints.** Edge (M90), ultra-long context, low-precision inference — these benefit from non-default choices.
3. **You're a frontier lab.** Spend a small fraction of your compute on Phase-X ablations before committing 100M GPU-hours to a run.
4. **You're debugging a regression.** If a new release is worse than the previous, ablation is how you isolate which change broke it.

**Don't ablate** when you're at small scale (< 10B params, < 1T tokens). The variance from data choice and tokenizer is bigger than architecture differences. Use Llama-3 defaults and ship.

> 🎓 **The Imad-Ch9 capstone.** Modern LLM architecture is **not magic** — it's the cumulative result of ~5-10 small, well-controlled ablations, each contributing 5-10% PPL or 2-5× inference speedup. Reading Llama-3 / Mistral / Qwen / DeepSeek papers with this lens turns mysterious choices into legible decisions. After M19/M20 (theory), M61 (GPT-2 → Llama 3 walk), and now M107 (ablation discipline), you can read any frontier architecture paper and **reproduce the ablation table** yourself.

## ✅ Recap

- **Ablation discipline** fixes everything but one variable; without it, claims confound.
- **PE**: **RoPE** is the 2026 default (Llama, Mistral, Qwen, DeepSeek); ALiBi extrapolates; NoPE is a research curiosity.
- **Attention**: **GQA-8** is the open default; **MLA** (DeepSeek) is the breakout; MQA for absolute fastest; FlashAttention-3 kernel for all.
- **KV-cache cost** dominates inference economics — MLA at 1/16 the cost of MHA enabled DeepSeek-V3's 10× cheaper API.
- **Activation**: **SwiGLU** is universal for >1B params; GELU for edge.
- **Normalization**: **Pre-LN + RMSNorm** universal; DyT (Zhu 2025) is the wild card.
- **Phases 1-4** stack the winners: baseline → +RoPE → +GQA → +SwiGLU → +RMSNorm = Llama-3 architecture.
- **Report 3 numbers**: same-FLOPs, same-parameters, same-wall-clock — they're not comparable.
- **Don't ablate at small scale** — use Llama-3 defaults and ship.

🎓 **Course complete. M1 → M107. Ship.**
